# Train SLT on PHOENIX-2014T (Colab)

Before running:
1. Upload the three pickles to Google Drive at `MyDrive/slt_data/`:
   - `phoenix14t.train.pickle.gzip`
   - `phoenix14t.dev.pickle.gzip`
   - `phoenix14t.test.pickle.gzip`
2. Set runtime to GPU (T4 is fine).
3. Run cells top to bottom. The trainer auto-resumes if Colab disconnects.

## 1. Mount Drive + check GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi

## 2. Clone this repo + install deps

In [ ]:
%cd /content
![ -d phoenix14t-slt-trainer ] || git clone https://github.com/AIVIETNAM-AIO-ThanhKieuVT/phoenix14t-slt-trainer.git
%cd phoenix14t-slt-trainer
!pip install -q -r requirements.txt

## 3. Set paths and verify data

We point directly at Drive for the data (no copy needed) and write the model to Drive
so it survives Colab disconnects.

In [ ]:
import os

DATA_DIR  = '/content/drive/MyDrive/slt_data'
MODEL_DIR = '/content/drive/MyDrive/models/slt_seed142_4layer_h384'
os.makedirs(MODEL_DIR, exist_ok=True)

for f in ('phoenix14t.train.pickle.gzip',
          'phoenix14t.dev.pickle.gzip',
          'phoenix14t.test.pickle.gzip'):
    p = f'{DATA_DIR}/{f}'
    assert os.path.exists(p), f'MISSING: {p}'
    print(f'  OK  {f}  ({os.path.getsize(p)/1e6:.1f} MB)')

ckpts = [f for f in os.listdir(MODEL_DIR) if f.endswith('.ckpt')]
print(f'\nExisting checkpoints in {MODEL_DIR}: {ckpts or "(none — fresh start)"}')

## 4. (Optional) Keep-alive heartbeat

Run this in a separate cell BEFORE training. Click the connect button every 5 min
to reduce idle disconnects. Has no effect on the training itself.

In [ ]:
from IPython.display import Javascript, display
display(Javascript('''
function ClickConnect(){
    console.log("Heartbeat:", new Date().toLocaleTimeString());
    document.querySelector("colab-toolbar-button#connect")?.click();
}
setInterval(ClickConnect, 5 * 60 * 1000);
'''))
print('Heartbeat installed (every 5 minutes).')

## 5. Train

Long-running. Tee the log to Drive so we can monitor / resume from any session.

In [ ]:
LOG_PATH = f'{MODEL_DIR}/training.log'
!mkdir -p $MODEL_DIR
!python train_slt.py \
    --train  $DATA_DIR/phoenix14t.train.pickle.gzip \
    --dev    $DATA_DIR/phoenix14t.dev.pickle.gzip \
    --test   $DATA_DIR/phoenix14t.test.pickle.gzip \
    --config config_seed142.yaml \
    --model_dir $MODEL_DIR \
    --device cuda 2>&1 | tee -a $LOG_PATH

## 6. Watch dev BLEU (run in a parallel cell while training)

In [ ]:
validations = f'{MODEL_DIR}/validations.txt'
if os.path.exists(validations):
    !tail -10 $validations
else:
    print(f'No validations yet at {validations}')

## 7. Verify final checkpoint

When dev BLEU-4 plateaus (typically ≥ 18), interrupt cell 5. Then:

In [ ]:
for f in ('best.ckpt', 'latest.ckpt', 'config.yaml',
          'gls.vocab', 'txt.vocab', 'validations.txt'):
    p = f'{MODEL_DIR}/{f}'
    if os.path.exists(p):
        print(f'  OK   {f:<20} {os.path.getsize(p)/1e6:>8.2f} MB')
    else:
        print(f'  ---  {f:<20} (missing)')

print()
print('Download best.ckpt back to local for downstream evaluation:')
print(f'  scp / drive-sync from {MODEL_DIR}/best.ckpt')

## What if Colab disconnects mid-training?

1. Re-open this notebook.
2. Run cells 1, 2, 3 (4 if you want the heartbeat).
3. Run cell 5 again — `train_slt.py` reads `latest.ckpt` from Drive and resumes.

No flags needed. The optimizer and scheduler state are restored from the checkpoint.